# 04 vLLM Serving Engine：从推理引擎到 Mac 本地运行

本章是 vLLM 专题。读完你应该能解释：vLLM 为什么快，PagedAttention 解决什么问题，scheduler 如何影响 TTFT/ITL，OpenAI-compatible server 如何接入业务，Mac 上为什么要用 vLLM-Metal，以及生产调优应该看哪些指标。

vLLM 是一个高性能 LLM inference/serving 框架。它不只是“启动一个模型 API”，而是围绕 KV cache、batching、调度、并发、OpenAI API、metrics 和部署做了一整套推理系统。


## 1. vLLM 在 AI Infra 栈中的位置

```mermaid
flowchart TB
    App["AI 应用 / AI App / Agent / RAG"] --> Gateway["模型网关 / LLM Gateway"]
    Gateway --> API["vLLM OpenAI 兼容 API 服务 / OpenAI-compatible API Server"]
    API --> Engine["vLLM 引擎核心 / Engine Core<br/>调度器 / scheduler + KV 缓存管理"]
    Engine --> Worker["模型工作进程 / Model Workers<br/>GPU / Metal 后端"]
    Worker --> Stream["流式 tokens / streaming tokens"]
    Engine --> Metrics["生产指标 / production metrics"]
```

业务代码最好不要直接依赖某个底层 engine 的内部接口，而是通过 OpenAI-compatible API 或网关调用。这样将来从 vLLM 切到 SGLang、云供应商或本地 MLX 时，业务改动更小。


## 2. vLLM V1 架构心智模型

vLLM V1 可以拆成几个角色：

- **API Server**：处理 HTTP、OpenAI-compatible endpoint、请求校验、tokenization、streaming response。
- **Engine Core**：核心调度循环，决定每一步处理哪些请求和 token。
- **Scheduler**：在 waiting/running 请求之间分配 token budget，平衡 prefill、decode、吞吐和延迟。
- **KV Cache Manager**：分配、回收、复用 KV cache blocks，支持 prefix caching。
- **Worker / Model Runner**：在 GPU/加速器上执行 forward pass。
- **Metrics**：暴露 queue time、TTFT、ITL、tokens、KV cache 使用率等生产指标。

```mermaid
flowchart LR
    Client["客户端 / client"] --> API["API 服务 / API Server"]
    API --> Queue["等待队列 / waiting queue"]
    Queue --> Scheduler["调度器 / Scheduler"]
    Scheduler --> KV["KV 缓存管理器 / KV Cache Manager"]
    Scheduler --> Runner["模型执行器 / Model Runner"]
    Runner --> Sampler["采样器 / Sampler"]
    Sampler --> API
    API --> Client
```


## 3. PagedAttention：为什么要把 KV cache 分块

传统做法如果为每个请求预留一整段最大上下文 KV cache，会产生大量浪费：多数请求不会真的用满最大长度，而且请求长度差异很大。PagedAttention 的核心思想类似操作系统分页：把 KV cache 切成固定大小 blocks，逻辑上连续，物理上可以分散。

```mermaid
flowchart TB
    ReqA["请求 A 逻辑 tokens / Request A logical tokens"] --> A1["块 1 / block 1"]
    ReqA --> A2["块 2 / block 2"]
    ReqB["请求 B 逻辑 tokens / Request B logical tokens"] --> B7["块 7 / block 7"]
    ReqC["请求 C 逻辑 tokens / Request C logical tokens"] --> C3["块 3 / block 3"]
    ReqC --> C9["块 9 / block 9"]
    Blocks["物理 KV 块池 / Physical KV block pool"] --> A1
    Blocks --> A2
    Blocks --> B7
    Blocks --> C3
    Blocks --> C9
```

收益：减少显存碎片和预留浪费，提高不同长度请求的并发承载能力，并为 prefix caching、block 复用和 eviction 提供基础。


## 4. Chunked prefill 与调优直觉

长 prompt 的 prefill 会占用大量 batch token budget。如果一次性处理一个超长 prompt，正在 decode 的请求可能被阻塞，用户看到流式输出卡顿。Chunked prefill 把大 prefill 拆成更小块，与 decode 混合调度。

`max_num_batched_tokens` 是一个非常关键的调参入口：

| 值偏小 | 值偏大 |
|---|---|
| decode 更容易被及时调度，ITL 可能更好 | prefill 一次处理更多 token，吞吐和 TTFT 可能更好 |
| 新请求长 prompt 可能 prefill 得更慢 | 长 prefill 可能影响 decode 流畅度 |

没有统一最优值。你需要根据 workload 的 prompt 长度、输出长度、并发和 SLA 做 benchmark。


In [ ]:
# KV cache 估算器：用于理解 max_model_len 和并发为什么危险。
def estimate_kv_cache_gib(num_layers, batch_size, seq_len, num_kv_heads, head_dim, dtype_bytes=2):
    bytes_total = num_layers * 2 * batch_size * seq_len * num_kv_heads * head_dim * dtype_bytes
    return bytes_total / (1024 ** 3)

for seq in [2048, 8192, 32768]:
    gib = estimate_kv_cache_gib(28, batch_size=8, seq_len=seq, num_kv_heads=8, head_dim=128)
    print(f'batch=8 seq={seq:5d} -> KV cache ≈ {gib:6.2f} GiB')


## 5. Offline inference 与 online serving

vLLM 常见两种使用方式：

- **Offline inference**：Python 中使用 `LLM` 和 `SamplingParams`，适合批量数据生成、评测、离线处理。
- **Online serving**：使用 `vllm serve` 启动 OpenAI-compatible server，适合业务服务接入。

离线推理关心总吞吐；在线服务还要关心 TTFT、ITL、排队时间、流式输出、限流、监控和稳定性。


In [ ]:
# 原生 vLLM 离线示例。Mac Apple Silicon 通常跳过，使用后面的 vLLM-Metal 路径。
import platform

if platform.system() == 'Darwin':
    print('当前是 macOS：跳过原生 vLLM 离线示例，请使用 vLLM-Metal + OpenAI SDK 路径。')
else:
    try:
        from vllm import LLM, SamplingParams
        prompts = ['解释 KV cache。', '解释 TTFT 和 ITL。']
        params = SamplingParams(temperature=0.2, top_p=0.95, max_tokens=128)
        llm = LLM(model='Qwen/Qwen2.5-1.5B-Instruct', generation_config='vllm', max_model_len=2048)
        outputs = llm.generate(prompts, params)
        for out in outputs:
            print(out.prompt, '->', out.outputs[0].text[:200])
    except ImportError as exc:
        print('未安装 vLLM；这是可选单元。', repr(exc))


## 6. OpenAI-compatible server 接入

典型启动命令：

```bash
vllm serve Qwen/Qwen2.5-1.5B-Instruct \
  --host 0.0.0.0 \
  --port 8000 \
  --api-key token-abc123 \
  --generation-config vllm \
  --max-model-len 2048
```

重要参数：

- `--generation-config vllm`：避免默认读取 Hugging Face 仓库的 generation_config，便于复现实验。
- `--max-model-len`：控制最大上下文，直接影响 KV cache 容量规划。
- `--api-key`：本地开发可以简单设置，生产应交给网关和密钥系统。
- `extra_body`：OpenAI SDK 可传 vLLM 扩展参数，例如 `top_k`、结构化输出相关参数等。


In [ ]:
from openai import OpenAI
import os

BASE_URL = os.environ.get('VLLM_BASE_URL', 'http://localhost:8000/v1')
API_KEY = os.environ.get('VLLM_API_KEY', 'token-abc123')
MODEL = os.environ.get('VLLM_MODEL', 'Qwen/Qwen2.5-1.5B-Instruct')

client = OpenAI(base_url=BASE_URL, api_key=API_KEY)
try:
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[{'role': 'user', 'content': '用三句话解释 vLLM 的价值。'}],
        temperature=0.2,
        max_tokens=128,
        extra_body={'top_k': 40},
    )
    print(resp.choices[0].message.content)
except Exception as exc:
    print('调用失败：请先启动 vLLM/vLLM-Metal server。')
    print('BASE_URL=', BASE_URL, 'MODEL=', MODEL)
    print(repr(exc))


## 7. Mac 与 vLLM-Metal

官方 vLLM 主路径主要面向 Linux/CUDA；Apple Silicon Mac 可使用 vLLM-Metal。它通过 MLX/Metal 后端让你在 Mac 上学习和跑小模型，但不要把它等同于生产 CUDA 主路径。

推荐安装路径：

```bash
curl -fsSL https://raw.githubusercontent.com/vllm-project/vllm-metal/main/install.sh | bash
source ~/.venv-vllm-metal/bin/activate
export VLLM_METAL_USE_PAGED_ATTENTION=1
export VLLM_METAL_MEMORY_FRACTION=auto
vllm serve mlx-community/Qwen2.5-1.5B-Instruct-4bit \
  --host 0.0.0.0 \
  --port 8000 \
  --api-key token-abc123 \
  --max-model-len 2048 \
  --generation-config vllm
```

差异和限制要点：

- 优先选择 `mlx-community/*-4bit` 小模型。
- 适合学习、本地开发、隐私场景，不适合高并发生产基准。
- multimodal、部分 CUDA 优化、特定量化/并行能力可能和主路径不同。
- 必须使用 native arm64 Python，Rosetta/x86_64 Python 不适合。


In [ ]:
# Mac/vLLM-Metal 环境检查：不安装、不修改环境。
import platform
from pathlib import Path

if platform.system() != 'Darwin':
    print('当前不是 macOS；跳过 vLLM-Metal 检查。')
elif platform.machine() != 'arm64':
    print('当前不是 native arm64 Python。请切换 arm64 Python 3.12。')
else:
    venv = Path.home() / '.venv-vllm-metal'
    print('Apple Silicon + arm64 Python: OK')
    print('vLLM-Metal venv exists:', venv.exists())
    print('vLLM CLI exists:', (venv / 'bin' / 'vllm').exists())


## 8. Metrics 与调优

生产 vLLM 服务要看指标，而不是只看单次 demo。关键指标包括：

| 指标 | 含义 | 常见定位 |
|---|---|---|
| request queue time | 请求等待调度时间 | 负载过高、实例不足、调度参数不合适 |
| time to first token | 首 token 时间 | 排队、tokenization、prefill、首步 decode |
| time per output token | 输出 token 间隔 | decode、KV cache、后端 kernel、batching |
| prompt/generation tokens | 输入/输出 token 数 | 成本、容量、异常长请求 |
| KV cache usage | KV cache 使用率 | OOM、preemption、上下文过长 |
| preemption | 请求被抢占/重算 | KV cache 压力或调度冲突 |

调优顺序建议：先固定 workload，再记录 baseline，然后一次只改一个参数。不要同时改模型、上下文、batch token budget 和并发，否则无法判断因果。


In [ ]:
# 轻量 streaming benchmark：需要本地 server，失败时只打印提示。
import time
from statistics import mean

prompts = ['解释 PagedAttention。', '解释 prefix caching。', '解释 chunked prefill。']

def stream_once(prompt, max_tokens=96):
    start = time.perf_counter()
    first = None
    chunks = []
    try:
        stream = client.chat.completions.create(
            model=MODEL,
            messages=[{'role': 'user', 'content': prompt}],
            temperature=0.2,
            max_tokens=max_tokens,
            stream=True,
        )
        for event in stream:
            delta = event.choices[0].delta.content or ''
            if delta and first is None:
                first = time.perf_counter()
            if delta:
                chunks.append(delta)
        end = time.perf_counter()
        return {'ok': True, 'ttft': None if first is None else first - start, 'latency': end - start, 'chars': len(''.join(chunks))}
    except Exception as exc:
        return {'ok': False, 'error': repr(exc)}

results = [stream_once(p) for p in prompts]
for r in results:
    print(r)
valid = [r for r in results if r.get('ok')]
if valid:
    print('avg_ttft=', mean(r['ttft'] for r in valid if r['ttft'] is not None))
    print('avg_latency=', mean(r['latency'] for r in valid))


## 9. 常见误区

- **只看 throughput，不看 TTFT/ITL。** 聊天产品最先影响体验的是首 token 和流式间隔。
- **盲目拉大 max_model_len。** 长上下文会显著增加 KV cache 预算。
- **把 Mac 本地结果当生产性能。** vLLM-Metal 是学习和本地开发路径，不代表 CUDA 集群表现。
- **不固定 generation_config。** 评测时采样和 generation config 不一致，会让结果不可复现。
- **没有 request id。** 没有 request id 就很难关联业务日志、网关日志、vLLM metrics 和用户反馈。


## 10. vLLM 参数与问题映射表

| 问题 | 相关参数/机制 | 调整思路 |
|---|---|---|
| OOM 或 KV cache 紧张 | `max_model_len`、并发、KV cache dtype、gpu memory utilization | 降上下文、限并发、换小模型、量化 KV cache |
| TTFT 高 | queue、prefill、`max_num_batched_tokens`、prefix cache | 看 prompt 长度和排队，优化检索压缩，调整 chunked prefill |
| ITL 高 | decode batch、KV cache 读取、后端 kernel | 降负载、调 batch token budget、看 GPU/Metal 后端 |
| 输出不复现 | generation config、temperature、top_p、seed | 固定采样参数，使用 `generation_config="vllm"` |
| 业务排障难 | request id、metrics、logs、traces | 网关和 vLLM 日志统一 request id |

调参前先定义 workload。没有 workload 的调参就是盲调。


## 11. 生产部署中 vLLM 和网关的边界

vLLM 负责高性能推理，但生产系统通常还需要网关和平台层：

```mermaid
flowchart LR
    App["业务应用"] --> GW["模型网关 / LLM Gateway<br/>鉴权 / 配额 / 路由 / fallback"]
    GW --> V1["vLLM 副本 A / replica A"]
    GW --> V2["vLLM 副本 B / replica B"]
    GW --> Cloud["云端降级 / cloud fallback"]
    V1 --> Metrics["Prometheus 指标 / metrics"]
    V2 --> Metrics
    GW --> Logs["请求日志与成本 / request logs / cost"]
```

不要把所有治理能力都塞进 vLLM。鉴权、租户预算、跨模型 fallback、A/B、审计和复杂策略更适合网关；KV cache、batching、scheduler、模型执行更适合 vLLM。


## 12. vLLM 面试表达

> vLLM 的核心价值是把 LLM 推理从“单请求模型调用”变成“高并发 token 调度系统”。它通过 PagedAttention 把 KV cache 分块管理，减少显存浪费；通过 continuous batching 和 chunked prefill 提高 GPU 利用率并平衡 TTFT/ITL；通过 prefix caching 复用共享 prompt；通过 OpenAI-compatible server 降低业务接入成本。生产调优时我会看 queue time、TTFT、ITL、tokens/sec、KV cache usage、preemption 和 OOM，而不是只看单条请求速度。


## 13. vLLM 上线检查清单

在把 vLLM 用作真实服务前，建议至少检查：

- 模型是否支持目标任务、chat template、工具调用或结构化输出。
- `max_model_len` 是否和业务需求、显存预算匹配。
- 是否固定 generation config 和 sampling 参数用于评测。
- 是否定义了最大 prompt tokens、最大 output tokens 和超长请求策略。
- 是否有 OpenAI-compatible client 和网关抽象，避免业务强耦合 vLLM 内部。
- 是否暴露 metrics，并接入 dashboard 和告警。
- 是否记录 request id、model、prompt version、token usage、latency breakdown。
- 是否做过真实 workload benchmark，而不是只跑单条 prompt。
- 是否定义 OOM、timeout、模型下载失败、fallback 和回滚策略。

这些检查能把“能启动”升级成“能负责”。


## 14. Mac 学习路径与云端生产路径

Mac/vLLM-Metal 很适合学习：你可以本地理解 OpenAI-compatible API、streaming、采样、max tokens、TTFT 近似测量、prompt 长度影响。但 Mac 路径不适合承担 CUDA 生产调优的全部结论。

建议双轨学习：

1. 在 Mac 上用 1.5B/3B 4bit 模型跑通 API 和 benchmark 客户端。
2. 用 Notebook 的公式估算 KV cache 和 token pressure。
3. 到云 GPU 上用同一批 prompts 跑 vLLM/SGLang benchmark。
4. 比较 Mac 和云端在 TTFT、ITL、吞吐、显存限制上的差异。
5. 把差异写成选型依据，而不是凭感觉说“本地够用”或“必须上云”。


## 15. 自测题

1. PagedAttention 和 prefix caching 分别解决什么问题？
2. `max_model_len` 为什么会影响显存，而不仅是输入校验？
3. `max_num_batched_tokens` 调大和调小分别可能影响什么指标？
4. 为什么生产中通常需要模型网关，而不是业务直接连 vLLM？
5. vLLM-Metal 适合什么场景？不适合什么场景？


## 来源与延伸阅读

本章正文已经把关键内容消化进教程，下面链接只用于延伸阅读和核对最新细节：vLLM、vLLM-Metal、Ray Serve LLM、SGLang、TensorRT-LLM、Text Generation Inference、LiteLLM、KServe、NVIDIA GPU Operator、OpenTelemetry GenAI。
